## Import the Model

In [1]:
import sys
sys.path.append('../')

# import src.TagModel as model
# import src.TagModel_lat as model_latency
import src.Auth as Auth
import utils.utils as utils
import numpy as np
import matplotlib.pyplot as plt 
import json
import os
import threading

### Run the Model

Just considering the E[A]

In [ ]:
m_nrs = [20,30]
t_nrs = [5,10,15]
p = [.95, .9, .85, .8, .75, .7]
q = [1]


def run_optimize(parameters):
    os.system("python3 optimize.py \'"+json.dumps(parameters)+"\'")

threads = []
for m_nr in m_nrs:
    for t_nr in t_nrs:
        for p_ in p:
            for q_ in q:
                parameters = {'m_nr': m_nr, 't_nr': t_nr, 
                              'p': p_, 'q': q_, 
                              'TagEveryMessage': True, 
                              'AtLeastOnce': False, 
                              'EquivalentA': True}
                threads.append(threading.Thread(target=run_optimize, args=(parameters,)))
                threads[-1].start() 
        cnt = 0
        for thread in threads:
            thread.join()
            cnt += 1
            print(cnt)
            
                # exp = utils.Run_Experiment(model        = model.math_model,
                #                            parameters   = parameters,
                #                            eval         = Auth.evaluate,
                #                            m_size       = 128,
                #                            t_size       = 256,
                #                            save         = True)


# parameters = {'m_nr': 25, 't_nr': 25, 
#               'p': 0.95, 'q': 1, 
#               'TagEveryMessage': True, 
#               'AtLeastOnce': False, 
#               'EquivalentA': True}

# exp = utils.Run_Experiment(model        = model.math_model,
#                            parameters   = parameters,
#                            eval         = Auth.evaluate,
#                            m_size       = 1024,
#                            t_size       = 256,
#                            save         = True)
# exp['eval']




### Looking at the saved Models

In [ ]:
exp = utils.Load_Experiments()

import seaborn as sns

for i in range(len(exp)):
    print(exp[i]['parameters'])
    print(exp[i]['eval'])
    sns.heatmap(exp[i]['results']['X'])
    plt.show()
    print('\n\n')

### Creating a  progressive MAC for comparison

In [2]:
plot = False

### AGG MAC x ###
def AGG_MAC_x(m_nr, x):
    X = np.zeros((m_nr, m_nr//x))
    for i in range(m_nr):
        X[i, i//x] = 1
    return X


for x in [2,3,4,5]:
    X = AGG_MAC_x(35-35%x,x)

    for i in [.6,.7, .8 , .9]:
        parameters = {'m_nr': X.shape[0], 't_nr': X.shape[1],
                        'p': i, 'q': 1,
                        'x': x,
                        'name': f'AggMAC_{x}'}
        exp = Auth.Create_Experiment(parameters,X= X)
        exp['eval'] = Auth.evaluate(exp,m_size=1024,t_size=256, security_requirments=256, plot=plot)
        if utils.Save_Experiment(exp) != True:
            print("Error saving experiment AggMAC" , "with parameters", parameters)

print("AGG MAC experiments done")

for i in [.7, .8 , .9]:
    parameters = {'m_nr': 20, 't_nr': 20,
                    'p': i, 'q': 1, 
                    'name': 'trad'}

    exp = Auth.Create_Experiment(parameters,X= np.eye(parameters['m_nr']))
    exp['eval'] = Auth.evaluate(exp,m_size=1024,t_size=256, security_requirments=256, plot=plot)
    if utils.Save_Experiment(exp) != True:
        print("Error saving experiment trad" , "with parameters", parameters)

    print("Traditional MAC experiments done")

    for x in [2,3,4,5]:

        parameters = {'m_nr': 20, 't_nr': 20,
                        'p': i, 'q': 1,
                        'x': x,
                        'name': f'Whip_{x}'}
        X = Auth.ProMAC_X(parameters['m_nr'],parameters['x'])
        exp = Auth.Create_Experiment(parameters,X= X)
        exp['eval'] = Auth.evaluate(exp,m_size=1024,t_size=256/parameters['x'], security_requirments=256, plot=plot)
        if utils.Save_Experiment(exp) != True:
            print("Error saving experiment ProMAC" , "with parameters", parameters)

Status: 1
Objective value: 17.0
Status: 1
Objective value: 17.0
Experiment saved as experiment number 7
Status: 1
Objective value: 17.0
Status: 1
Objective value: 17.0
Experiment saved as experiment number 8
Status: 1
Objective value: 17.0
Status: 1
Objective value: 17.0
Experiment saved as experiment number 9
Status: 1
Objective value: 17.0
Status: 1
Objective value: 17.0
Experiment saved as experiment number 10
Status: 1
Objective value: 11.0
Status: 1
Objective value: 11.0
Experiment saved as experiment number 11
Status: 1
Objective value: 11.0
Status: 1
Objective value: 11.0
Experiment saved as experiment number 12
Status: 1
Objective value: 11.0
Status: 1
Objective value: 11.0
Experiment saved as experiment number 13
Status: 1
Objective value: 11.0
Status: 1
Objective value: 11.0
Experiment saved as experiment number 14
Status: 1
Objective value: 8.0
Status: 1
Objective value: 8.0
Experiment saved as experiment number 15
Status: 1
Objective value: 8.0
Status: 1
Objective value: 8.

In [ ]:
experiments = utils.Load_Experiments(filePath='Xs (1).pkl')
for exp in experiments:
    print(experiments[exp]['parameters'])
    print(experiments[exp]['eval'])
    plt.imshow(experiments[exp]['results']['X'])
    plt.show()
    print('\n\n')

### Run the optimizer With Latency

In [ ]:
parameters = {'m_nr': 10, 't_nr': 10,
                'p': 0.9, 'q': .9,
                'TagEveryMessage': True,
                'AtLeastOnce': False,
                'EquivalentA': True,
                'weight_A': 1,
                'weight_L': 0}


for i in range(10):
    exp = utils.Run_Experiment(model = model_latency.math_model,
                            parameters = parameters,
                            eval=Auth.evaluate,
                            save=True,
                            m_size=1024,
                            t_size=256)
    parameters['weight_L'] += 0.1
    print(exp['eval'])

# exp['eval']